In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
import pandas as pd
# point this at the directory *containing* tpch/
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))  
sys.path.insert(0, tpch_parent)

#from tpch import load_part, load_partsupp, load_supplier, load_nation,load_region,  q02
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15" #"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}  # or load from JSON

In [4]:

### cell 0 ###

# load just the datasets q01 needs:
def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)


In [5]:

### cell 1 ###

def load_partsupp(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/partsupp.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")  
    return df
partsupp = load_partsupp(DATA_ROOT, **STORAGE_OPTS)


In [6]:

### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)


In [7]:

### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)


In [8]:

### cell 4 ###

def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/region.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [9]:
%%time
# ### cell 5 ###

nation_filtered = nation.loc[:, ["N_NATIONKEY", "N_NAME", "N_REGIONKEY"]]
region_filtered = region[(region["R_NAME"] == "EUROPE")]
region_filtered = region_filtered.loc[:, ["R_REGIONKEY"]]
r_n_merged = nation_filtered.merge(
    region_filtered, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
)
r_n_merged = r_n_merged.loc[:, ["N_NATIONKEY", "N_NAME"]]
supplier_filtered = supplier.loc[
    :,
    [
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_NATIONKEY",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
s_r_n_merged = r_n_merged.merge(
    supplier_filtered, left_on="N_NATIONKEY", right_on="S_NATIONKEY", how="inner"
)
s_r_n_merged = s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]
ps_s_r_n_merged = s_r_n_merged.merge(
    partsupp_filtered, left_on="S_SUPPKEY", right_on="PS_SUPPKEY", how="inner"
)
ps_s_r_n_merged = ps_s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_PARTKEY",
        "PS_SUPPLYCOST",
    ],
]
part_filtered = part.loc[:, ["P_PARTKEY", "P_MFGR", "P_SIZE", "P_TYPE"]]
part_filtered = part_filtered[
    (part_filtered["P_SIZE"] == 15)
    & (part_filtered["P_TYPE"].str.endswith("BRASS"))
]
part_filtered = part_filtered.loc[:, ["P_PARTKEY", "P_MFGR"]]
merged_df = part_filtered.merge(
    ps_s_r_n_merged, left_on="P_PARTKEY", right_on="PS_PARTKEY", how="inner"
)
merged_df = merged_df.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_SUPPLYCOST",
        "P_PARTKEY",
        "P_MFGR",
    ],
]
min_values = merged_df.groupby("P_PARTKEY", as_index=False, sort=False)[
    "PS_SUPPLYCOST"
].min()
min_values.columns = ["P_PARTKEY_CPY", "MIN_SUPPLYCOST"]
merged_df = merged_df.merge(
    min_values,
    left_on=["P_PARTKEY", "PS_SUPPLYCOST"],
    right_on=["P_PARTKEY_CPY", "MIN_SUPPLYCOST"],
    how="inner",
)
total = merged_df.loc[
    :,
    [
        "S_ACCTBAL",
        "S_NAME",
        "N_NAME",
        "P_PARTKEY",
        "P_MFGR",
        "S_ADDRESS",
        "S_PHONE",
        "S_COMMENT",
    ],
]
total = total.sort_values(
    by=["S_ACCTBAL", "N_NAME", "S_NAME", "P_PARTKEY"],
    ascending=[False, True, True, True],
)

CPU times: user 299 ms, sys: 61.8 ms, total: 361 ms
Wall time: 365 ms


In [10]:
%%time
#original
nation_filtered = nation.loc[:, ["N_NATIONKEY", "N_NAME", "N_REGIONKEY"]]
region_filtered = region[(region["R_NAME"] == "EUROPE")]
region_filtered = region_filtered.loc[:, ["R_REGIONKEY"]]
r_n_merged = nation_filtered.merge(
    region_filtered, left_on="N_REGIONKEY", right_on="R_REGIONKEY", how="inner"
)
r_n_merged = r_n_merged.loc[:, ["N_NATIONKEY", "N_NAME"]]
supplier_filtered = supplier.loc[
    :,
    [
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_NATIONKEY",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
s_r_n_merged = r_n_merged.merge(
    supplier_filtered, left_on="N_NATIONKEY", right_on="S_NATIONKEY", how="inner"
)
s_r_n_merged = s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_SUPPKEY",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
    ],
]
partsupp_filtered = partsupp.loc[:, ["PS_PARTKEY", "PS_SUPPKEY", "PS_SUPPLYCOST"]]
ps_s_r_n_merged = s_r_n_merged.merge(
    partsupp_filtered, left_on="S_SUPPKEY", right_on="PS_SUPPKEY", how="inner"
)
ps_s_r_n_merged = ps_s_r_n_merged.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_PARTKEY",
        "PS_SUPPLYCOST",
    ],
]
part_filtered = part.loc[:, ["P_PARTKEY", "P_MFGR", "P_SIZE", "P_TYPE"]]
part_filtered = part_filtered[
    (part_filtered["P_SIZE"] == 15)
    & (part_filtered["P_TYPE"].str.endswith("BRASS"))
]
part_filtered = part_filtered.loc[:, ["P_PARTKEY", "P_MFGR"]]
merged_df = part_filtered.merge(
    ps_s_r_n_merged, left_on="P_PARTKEY", right_on="PS_PARTKEY", how="inner"
)
merged_df = merged_df.loc[
    :,
    [
        "N_NAME",
        "S_NAME",
        "S_ADDRESS",
        "S_PHONE",
        "S_ACCTBAL",
        "S_COMMENT",
        "PS_SUPPLYCOST",
        "P_PARTKEY",
        "P_MFGR",
    ],
]
min_values = merged_df.groupby("P_PARTKEY", as_index=False, sort=False)[
    "PS_SUPPLYCOST"
].min()
min_values.columns = ["P_PARTKEY_CPY", "MIN_SUPPLYCOST"]
merged_df = merged_df.merge(
    min_values,
    left_on=["P_PARTKEY", "PS_SUPPLYCOST"],
    right_on=["P_PARTKEY_CPY", "MIN_SUPPLYCOST"],
    how="inner",
)
total = merged_df.loc[
    :,
    [
        "S_ACCTBAL",
        "S_NAME",
        "N_NAME",
        "P_PARTKEY",
        "P_MFGR",
        "S_ADDRESS",
        "S_PHONE",
        "S_COMMENT",
    ],
]
total = total.sort_values(
    by=["S_ACCTBAL", "N_NAME", "S_NAME", "P_PARTKEY"],
    ascending=[False, True, True, True],
)

CPU times: user 343 ms, sys: 32 ms, total: 375 ms
Wall time: 432 ms


In [11]:
total, 432 

,S_ACCTBAL,S_NAME,N_NAME,P_PARTKEY,P_MFGR,S_ADDRESS,S_PHONE,S_COMMENT
908,9994.37,Supplier#000030084,GERMANY,380077,Manufacturer#5,"gBEvSkyW o1uHea0CV,oHtkTTVW",17-519-171-6883,pinto beans sleep fluffily alongside of the sl...
1995,9992.54,Supplier#000099650,RUSSIA,824625,Manufacturer#3,ySI FMlh9gHkEDN6gQWf3,32-971-481-2533,ged deposits cajole carefully packages. carefu...
944,9987.51,Supplier#000020657,ROMANIA,395653,Manufacturer#5,"4pL,8BT3Yun,17QHqAr9 A,ZFyyuH4L",29-167-460-7830,otes. excuses behind the blithely regular pack...
4197,9986.40,Supplier#000082995,RUSSIA,1782994,Manufacturer#5,CXiBNZ6DUBjgY,32-510-919-3096,"nding instructions boost. unusual, regular asy..."
1525,9984.69,Supplier#000008875,ROMANIA,633856,Manufacturer#4,"hRdOqKqyU,sHq",29-132-904-4395,ong the bold pinto beans are furiously blithel...
...,...,...,...,...,...,...,...,...
2153,-995.84,Supplier#000074505,GERMANY,899480,Manufacturer#5,tFJX5iGa2fesD125u6zVTC4Tnxd0661r,17-502-851-2789,bold instructions are carefully carefully reg...
2280,-995.84,Supplier#000074505,GERMANY,949495,Manufacturer#5,tFJX5iGa2fesD125u6zVTC4Tnxd0661r,17-502-851-2789,bold instructions are carefully carefully reg...
2868,-998.37,Supplier#000034531,GERMANY,1209518,Manufacturer#2,L7UOHUBxtK,17-558-874-3467,lyly. regular platelets wake slyly atop the bl...
3103,-998.37,Supplier#000034531,GERMANY,1309517,Manufacturer#5,L7UOHUBxtK,17-558-874-3467,lyly. regular platelets wake slyly atop the bl...


In [11]:
total

,S_ACCTBAL,S_NAME,N_NAME,P_PARTKEY,P_MFGR,S_ADDRESS,S_PHONE,S_COMMENT
908,9994.37,Supplier#000030084,GERMANY,380077,Manufacturer#5,"gBEvSkyW o1uHea0CV,oHtkTTVW",17-519-171-6883,pinto beans sleep fluffily alongside of the sl...
1995,9992.54,Supplier#000099650,RUSSIA,824625,Manufacturer#3,ySI FMlh9gHkEDN6gQWf3,32-971-481-2533,ged deposits cajole carefully packages. carefu...
944,9987.51,Supplier#000020657,ROMANIA,395653,Manufacturer#5,"4pL,8BT3Yun,17QHqAr9 A,ZFyyuH4L",29-167-460-7830,otes. excuses behind the blithely regular pack...
4197,9986.40,Supplier#000082995,RUSSIA,1782994,Manufacturer#5,CXiBNZ6DUBjgY,32-510-919-3096,"nding instructions boost. unusual, regular asy..."
1525,9984.69,Supplier#000008875,ROMANIA,633856,Manufacturer#4,"hRdOqKqyU,sHq",29-132-904-4395,ong the bold pinto beans are furiously blithel...
...,...,...,...,...,...,...,...,...
2153,-995.84,Supplier#000074505,GERMANY,899480,Manufacturer#5,tFJX5iGa2fesD125u6zVTC4Tnxd0661r,17-502-851-2789,bold instructions are carefully carefully reg...
2280,-995.84,Supplier#000074505,GERMANY,949495,Manufacturer#5,tFJX5iGa2fesD125u6zVTC4Tnxd0661r,17-502-851-2789,bold instructions are carefully carefully reg...
2868,-998.37,Supplier#000034531,GERMANY,1209518,Manufacturer#2,L7UOHUBxtK,17-558-874-3467,lyly. regular platelets wake slyly atop the bl...
3103,-998.37,Supplier#000034531,GERMANY,1309517,Manufacturer#5,L7UOHUBxtK,17-558-874-3467,lyly. regular platelets wake slyly atop the bl...


In [10]:
### cell 6 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 4667 entries, 908 to 3246
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   S_ACCTBAL  4667 non-null   float64
 1   S_NAME     4667 non-null   object
 2   N_NAME     4667 non-null   object
 3   P_PARTKEY  4667 non-null   int64
 4   P_MFGR     4667 non-null   object
 5   S_ADDRESS  4667 non-null   object
 6   S_PHONE    4667 non-null   object
 7   S_COMMENT  4667 non-null   object
dtypes: float64(1), int64(1), object(6)
memory usage: 869.0+ KB
